[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tunnel-ai/way/blob/main/notebooks/solutions_03_01_exercise_guided.ipynb)

In [1]:
# --- Course setup (uncomment when running on Colab) ---
#!git clone https://github.com/tunnel-ai/way.git
#import sys; sys.path.insert(0, "/content/way/src")

# Module 3 — Classification (Guided Exercise — Solution)

**Solution to** `03_01_exercise_guided.ipynb`.

**Target:** `is_fraud` (binary, ~4% prevalence).

All TODOs filled in, plus answers to the check-in questions at the end.

In [2]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    roc_auc_score,
    average_precision_score,
    precision_recall_curve,
)

RANDOM_STATE = 1955
np.random.seed(RANDOM_STATE)

In [3]:
from core.generators.transaction_risk_dgp import generate_transaction_risk_dataset

df = generate_transaction_risk_dataset(seed=RANDOM_STATE)

print(df.shape)
df.head()

(126521, 27)


,transaction_id,account_id,account_age_days,merchant_id,merchant_name,merchant_category,merchant_risk_score,payment_channel,country,is_foreign_transaction,...,prior_fraud_count_90d,transaction_amount,avg_transaction_amount_30d,std_transaction_amount_30d,merchant_description,is_fraud,transaction_loss_amount,chargeback_flag,manual_review_score,fraud_probability_latent
0,1,1,3172,3082,MERCH_3082,education,-0.0614,online,US,0,...,0,4.69,12.25,9.82,MERCH_3082 education ONLINE -X,0,0.0,0,0.340,0.0275
1,2,1,3172,3,WALMART,charity,-0.3005,card_present,US,0,...,0,14.54,7.56,11.20,NaN,0,0.0,0,0.000,0.0204
2,3,1,3172,1,AMAZON,rideshare,0.0271,card_present,US,0,...,0,12.64,11.76,5.07,AMAZON rideshare POS CO,0,0.0,0,0.000,0.0267
3,4,1,3172,12,TARGET,utilities,-0.2635,online,US,0,...,0,8.41,9.74,17.60,TARGET utilities ONLINE -X,0,0.0,0,0.113,0.0155
4,5,1,3172,51,MERCH_0051,utilities,-0.0575,mobile,US,0,...,0,9.84,8.41,4.40,MERCH_0051 utilities MOB PAY,0,0.0,0,0.109,0.0101


## 1) Define X / y with leakage hygiene, then split

In [4]:
TARGET = "is_fraud"

DROP_COLS = [
    TARGET,
    # Leakage / post-event fields
    "transaction_loss_amount",
    "chargeback_flag",
    "manual_review_score",
    "fraud_probability_latent",
    # Pure IDs
    "transaction_id",
    "account_id",
    # Redundant / high-cardinality text
    "merchant_description",
    "merchant_name",
]

X = df.drop(columns=DROP_COLS).copy()
X["merchant_id"] = X["merchant_id"].astype(str)

y = df[TARGET].astype(int)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

print("Train fraud rate:", y_train.mean())
print("Valid fraud rate:", y_valid.mean())
print("X_train shape:  ", X_train.shape)

Train fraud rate: 0.04013067762672568
Valid fraud rate: 0.04011887072808321
X_train shape:   (94890, 18)


## 2) Preprocessing pipeline (provided)

In [5]:
high_card_cols = ["merchant_id"]

categorical_cols = X_train.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
low_card_cols = [c for c in categorical_cols if c not in high_card_cols]
numeric_cols = [c for c in X_train.columns if c not in categorical_cols]

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])
low_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])
high_card_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", min_frequency=50)),
])

preprocess = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat_low", low_card_transformer, low_card_cols),
        ("cat_high", high_card_transformer, high_card_cols),
    ],
    remainder="drop",
)

## 3) Fit logistic regression and evaluate

In [6]:
logit = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    )),
])

logit.fit(X_train, y_train)
y_proba = logit.predict_proba(X_valid)[:, 1]

print("ROC-AUC:", roc_auc_score(y_valid, y_proba))
print("PR-AUC: ", average_precision_score(y_valid, y_proba))

ROC-AUC: 0.6421069657548066
PR-AUC:  0.08477478688378677


## 4) Choose an operating threshold via expected cost

In [7]:
C_FN = 50
C_FP = 1

precision, recall, thresholds = precision_recall_curve(y_valid, y_proba)
precision = precision[:-1]
recall = recall[:-1]

costs = []
for t in thresholds:
    y_hat = (y_proba >= t).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_valid, y_hat).ravel()
    costs.append(C_FN * fn + C_FP * fp)
costs = np.array(costs)

best_idx = costs.argmin()
best_t = thresholds[best_idx]
y_pred_best = (y_proba >= best_t).astype(int)

print("Best threshold:", best_t)
print("Min expected cost:", costs[best_idx])
print(confusion_matrix(y_valid, y_pred_best))
print(classification_report(y_valid, y_pred_best, digits=4))

Best threshold: 0.16255604709281163
Min expected cost: 30233
[[  279 30083]
 [    3  1266]]
              precision    recall  f1-score   support

           0     0.9894    0.0092    0.0182     30362
           1     0.0404    0.9976    0.0776      1269

    accuracy                         0.0488     31631
   macro avg     0.5149    0.5034    0.0479     31631
weighted avg     0.9513    0.0488    0.0206     31631



## 5) Engineer features and measure the lift

In [8]:
def engineer_features(X):
    X = X.copy()
    # 3-way interaction: foreign + new device + online channel
    X["foreign_new_device_online"] = (
        X["is_foreign_transaction"]
        * X["is_new_device"]
        * (X["payment_channel"] == "online").astype(int)
    )
    # Hour-of-day binning
    night_hours = {0, 1, 2, 3, 4, 5, 21, 22, 23}
    X["is_night"] = X["hour_of_day"].isin(night_hours).astype(int)
    # Log transforms for heavy-tailed numerics
    X["log_amount"] = np.log1p(X["transaction_amount"])
    X["log_velocity_24h"] = np.log1p(X["transactions_last_24h"])
    return X

X_train_eng = engineer_features(X_train)
X_valid_eng = engineer_features(X_valid)

numeric_cols_eng = numeric_cols + [
    "foreign_new_device_online",
    "is_night",
    "log_amount",
    "log_velocity_24h",
]

print("X_train_eng shape:", X_train_eng.shape)
print("New columns:", X_train_eng.columns[-4:].tolist())

X_train_eng shape: (94890, 22)
New columns: ['foreign_new_device_online', 'is_night', 'log_amount', 'log_velocity_24h']


### Refit and compare

In [9]:
preprocess_eng = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols_eng),
        ("cat_low", low_card_transformer, low_card_cols),
        ("cat_high", high_card_transformer, high_card_cols),
    ],
    remainder="drop",
)

logit_eng = Pipeline(steps=[
    ("preprocess", preprocess_eng),
    ("model", LogisticRegression(
        max_iter=2000,
        class_weight="balanced",
        solver="liblinear",
        random_state=RANDOM_STATE,
    )),
])

logit_eng.fit(X_train_eng, y_train)
y_proba_eng = logit_eng.predict_proba(X_valid_eng)[:, 1]

rows = []
for name, proba in [("Raw", y_proba), ("Engineered", y_proba_eng)]:
    rows.append({
        "model": name,
        "ROC_AUC": roc_auc_score(y_valid, proba),
        "PR_AUC":  average_precision_score(y_valid, proba),
    })
pd.DataFrame(rows).round(4)

,model,ROC_AUC,PR_AUC
0,Raw,0.6421,0.0848
1,Engineered,0.6650,0.1138


## 6) Check in — answers

**1. What ROC-AUC and PR-AUC did your raw logistic regression achieve? Why is PR-AUC the more honest summary at a 4% base rate?**

Raw logit lands at **ROC-AUC ≈ 0.642** and **PR-AUC ≈ 0.085**. ROC-AUC looks fine until you realize that *random* guessing has ROC = 0.50, so 0.64 is only a modest lift. PR-AUC is more honest because the baseline for a random classifier is the class prevalence itself (~0.04). Our 0.085 is roughly 2× baseline — measurable, but far from impressive. PR-AUC also weights performance in the high-precision regime, which is where rare-event detectors actually operate.

**2. What threshold did the cost minimization pick? Roughly how many false alarms does that produce per fraud caught?**

The cost-minimizing threshold is around **0.16**, far below the default 0.50. With C_FN = 50 and C_FP = 1, missing fraud is so expensive that the model essentially flags everything with non-trivial fraud probability — confusion matrix shows ~30,000 false positives for ~1,260 true positives caught, i.e., **~24 false alarms per fraud caught.** That sounds extreme, but at the chosen cost ratio it's optimal: avoiding one $50 fraud loss is worth absorbing 50 separate $1 manual reviews. In a real system, you'd revisit C_FN and C_FP based on actual operational data before shipping anything this aggressive.

**3. How much did engineering lift PR-AUC? Why did adding `foreign_new_device_online` help logistic regression specifically?**

PR-AUC goes from ~0.085 to ~0.114 (**+34% relative**), and ROC-AUC from ~0.642 to ~0.665. The interaction column is the workhorse: when all three of `is_foreign_transaction`, `is_new_device`, and `payment_channel == "online"` fire together (about 1.1% of transactions), the fraud rate is **~30% — almost 8× baseline.** Logistic regression cannot represent that pattern from raw columns because it's a multiplicative effect, and logit only sums standardized inputs. Adding the explicit AND column lets the model learn a single coefficient that captures the spike directly.

**4. The main notebook fits the same engineered features to a Random Forest and sees almost no lift. Why does engineering help logit but not RF?**

Random Forests build trees by recursively splitting on features — *that's exactly how interactions are represented in trees.* If splits are taken in order (`is_foreign_transaction == 1`) → (`is_new_device == 1`) → (`payment_channel == online`), the leaf at the bottom of that path **is** the 3-way interaction. RF doesn't need a hand-engineered column for it. Linear models do, because they cannot compose features.

The general principle: **feature engineering pays off most for the model class that can't learn the structure on its own.** If you're already using gradient boosting or RF, your time is usually better spent on *new* signal (different data sources, longer history, additional context) than on engineered combinations of features the model can already see.